# 03 · Geo analysis: where are Paris apartments expensive?

1. Choropleth maps of median price per m² by arrondissement (geopandas + matplotlib)
2. Distance to the nearest metro/RER station and to the city centre, per sale (`src/features.py`, Lambert-93, metres)
3. Interactive folium map → `reports/paris_map.html`

> Run first: `python src/download_data.py` (boundaries and stations), `python src/build_database.py`, then `python src/features.py`.
>
> **Privacy:** DVF reuse rules forbid re-identifying people. Maps show arrondissement aggregates; the interactive map shows only a random sample of sales, with coordinates blurred by about 100 m and prices rounded.

In [ ]:
from pathlib import Path

import duckdb
import folium
import geopandas as gpd
import matplotlib.patheffects as pe
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick
import numpy as np
import pandas as pd
import seaborn as sns
from branca.colormap import LinearColormap

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
DB_PATH = PROJECT_ROOT / "data" / "paris_dvf.duckdb"
EXTERNAL_DIR = PROJECT_ROOT / "data" / "external"
PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"
FIG_DIR = PROJECT_ROOT / "reports" / "figures"
FIG_DIR.mkdir(parents=True, exist_ok=True)

BLUE = "#3b6ea5"
sns.set_theme(style="whitegrid")

con = duckdb.connect(str(DB_PATH), read_only=True)
q = lambda sql: con.execute(sql).df()

## 1. Arrondissement boundaries

In [ ]:
arr = gpd.read_file(EXTERNAL_DIR / "arrondissements.geojson")
cols = {c.lower(): c for c in arr.columns}
if "c_ar" in cols:
    arr["arrondissement"] = arr[cols["c_ar"]].astype(int)
else:
    arr["arrondissement"] = arr[cols["c_arinsee"]].astype(int) - 75100
arr = arr[["arrondissement", "geometry"]].sort_values("arrondissement").reset_index(drop=True)
print(f"{len(arr)} arrondissements, CRS {arr.crs}")

## 2. Choropleth: median price per m² and change over the period

In [ ]:
medians = q("""
    SELECT arrondissement, year, median_price_m2
    FROM agg_median_by_arr_year
""")
first_year, last_year = medians["year"].min(), medians["year"].max()
latest = medians[medians["year"] == last_year][["arrondissement", "median_price_m2"]]
change = q("SELECT arrondissement, change_pct FROM agg_period_change")

arr_stats = arr.merge(latest, on="arrondissement").merge(change, on="arrondissement")
arr_proj = arr_stats.to_crs("EPSG:2154")  # projected CRS so Paris isn't stretched

fig, axes = plt.subplots(1, 2, figsize=(16, 7))
arr_proj.plot(column="median_price_m2", cmap="Blues", linewidth=0.6, edgecolor="white", legend=True,
              legend_kwds={"label": "€ per m²", "shrink": 0.6}, ax=axes[0])
axes[0].set_title(f"Median price per m², {last_year}")

lim = arr_proj["change_pct"].abs().max()
arr_proj.plot(column="change_pct", cmap="RdBu", vmin=-lim, vmax=lim, linewidth=0.6, edgecolor="white", legend=True,
              legend_kwds={"label": "% change", "shrink": 0.6}, ax=axes[1])
axes[1].set_title(f"Change in median price per m², {first_year} → {last_year}")

for ax in axes:
    ax.set_axis_off()
    for _, row in arr_proj.iterrows():
        p = row.geometry.representative_point()
        ax.annotate(str(row["arrondissement"]), (p.x, p.y), ha="center", va="center", fontsize=9,
                    path_effects=[pe.withStroke(linewidth=2.5, foreground="white")])

fig.tight_layout()
fig.savefig(FIG_DIR / "map_median_price_m2.png", dpi=150, bbox_inches="tight")
plt.show()
arr_stats.drop(columns="geometry").sort_values("median_price_m2", ascending=False)

**Insight:** _to be written from the real output above._

## 3. Distance to metro/RER and to the centre

`src/features.py` finds the nearest metro or RER station for every sale with a spatial nearest-neighbour join in Lambert-93 (metres), and the distance to point zero in front of Notre-Dame.

In [ ]:
geo = q("""
    SELECT s.arrondissement, s.price_per_m2, s.surface_reelle_bati, f.dist_metro_m, f.dist_centre_m
    FROM clean_sales AS s
    JOIN sales_features AS f USING (id_mutation)
""")
print(f"Sales with geo features: {len(geo):,}")
geo[["dist_metro_m", "dist_centre_m"]].describe(percentiles=[0.1, 0.25, 0.5, 0.75, 0.9]).round(0)

In [ ]:
geo["metro_bucket"] = pd.cut(geo["dist_metro_m"], [0, 150, 300, 450, 600, np.inf],
                             labels=["< 150 m", "150-300 m", "300-450 m", "450-600 m", "600 m +"], right=False)
geo["centre_bucket"] = pd.cut(geo["dist_centre_m"] / 1000, [0, 1, 2, 3, 4, 5, 6, np.inf],
                              labels=["< 1 km", "1-2 km", "2-3 km", "3-4 km", "4-5 km", "5-6 km", "6 km +"], right=False)

by_metro = geo.groupby("metro_bucket", observed=True)["price_per_m2"].agg(sales="size", median_price_m2="median").round(0)
by_centre = geo.groupby("centre_bucket", observed=True)["price_per_m2"].agg(sales="size", median_price_m2="median").round(0)

fig, axes = plt.subplots(1, 2, figsize=(14, 4))
axes[0].bar(by_metro.index.astype(str), by_metro["median_price_m2"], color=BLUE)
axes[0].set_title("Median €/m² by distance to nearest metro/RER")
axes[1].bar(by_centre.index.astype(str), by_centre["median_price_m2"], color=BLUE)
axes[1].set_title("Median €/m² by distance to the centre")
for ax in axes:
    ax.yaxis.set_major_formatter(mtick.FuncFormatter(lambda x, _: f"{x:,.0f} €"))
fig.tight_layout(); fig.savefig(FIG_DIR / "geo_distance_buckets.png", dpi=150); plt.show()

display(by_metro); display(by_centre)

Distance to the centre and distance to the metro are linked: the centre has the densest network. To see the metro effect on its own, compare sales **within the same arrondissement**:

In [ ]:
# Price premium of being < 300 m from a station vs 300 m+, within each arrondissement.
geo["near_metro"] = np.where(geo["dist_metro_m"] < 300, "near (< 300 m)", "far (300 m +)")
within = (geo.groupby(["arrondissement", "near_metro"])["price_per_m2"].median().unstack())
within["premium_pct"] = (100 * (within["near (< 300 m)"] / within["far (300 m +)"] - 1)).round(1)
print(f"Median premium for being < 300 m from metro/RER, within arrondissement: {within['premium_pct'].median():+.1f}%")
print(f"Arrondissements where near is more expensive: {(within['premium_pct'] > 0).sum()} of {within['premium_pct'].notna().sum()}")
within.round(0)

**Insight:** _to be written from the real output above._

## 4. Interactive map

A random sample of 3,000 sales from the last year, coloured by price per m², on top of the arrondissement medians.
Coordinates are blurred by up to ~100 m and prices rounded to 100 €/m², so no individual sale can be located.

In [ ]:
rng = np.random.default_rng(42)
sample = q(f"""
    SELECT arrondissement, price_per_m2, surface_reelle_bati, longitude, latitude
    FROM clean_sales
    WHERE year = {last_year} AND longitude IS NOT NULL
    USING SAMPLE 3000 ROWS (reservoir, 42)
""")
# Blur: ~0.0009° latitude ≈ 100 m; ~0.0013° longitude ≈ 100 m at Paris latitude.
sample["lat"] = sample["latitude"] + rng.uniform(-0.0009, 0.0009, len(sample))
sample["lon"] = sample["longitude"] + rng.uniform(-0.0013, 0.0013, len(sample))
sample["price_m2"] = (sample["price_per_m2"] / 100).round() * 100

m = folium.Map(location=[48.8566, 2.3522], zoom_start=12, tiles="OpenStreetMap")

folium.Choropleth(
    geo_data=arr_stats[["arrondissement", "median_price_m2", "geometry"]].to_json(),
    data=arr_stats, columns=["arrondissement", "median_price_m2"], key_on="feature.properties.arrondissement",
    fill_color="Blues", fill_opacity=0.35, line_opacity=0.6,
    legend_name=f"Median € per m² by arrondissement, {last_year}", name="Arrondissement median",
).add_to(m)

lo, hi = sample["price_m2"].quantile([0.05, 0.95])
cmap = LinearColormap(["#2c7bb6", "#ffffbf", "#d7191c"], vmin=lo, vmax=hi, caption="Sale price per m² (sample)")
points = folium.FeatureGroup(name="Sample of sales")
for r in sample.itertuples():
    folium.CircleMarker(
        location=[r.lat, r.lon], radius=3, weight=0, fill=True, fill_opacity=0.75,
        fill_color=cmap(min(max(r.price_m2, lo), hi)),
        tooltip=f"{r.arrondissement}e · {r.price_m2:,.0f} €/m² · {r.surface_reelle_bati:.0f} m²",
    ).add_to(points)
points.add_to(m)
cmap.add_to(m)
folium.LayerControl().add_to(m)

out = PROJECT_ROOT / "reports" / "paris_map.html"
m.save(out)
print(f"Saved {out}")
m

## 5. Export for Power BI

In [ ]:
geo_by_arr = geo.groupby("arrondissement").agg(
    sales=("price_per_m2", "size"),
    median_dist_metro_m=("dist_metro_m", "median"),
    median_dist_centre_m=("dist_centre_m", "median"),
    share_within_300m_metro_pct=("dist_metro_m", lambda d: 100 * (d < 300).mean()),
).round(1).reset_index()
geo_by_arr.to_csv(PROCESSED_DIR / "agg_geo_by_arr.csv", index=False)
by_metro.reset_index().to_csv(PROCESSED_DIR / "agg_price_by_metro_distance.csv", index=False)
by_centre.reset_index().to_csv(PROCESSED_DIR / "agg_price_by_centre_distance.csv", index=False)
geo_by_arr

## Key takeaways

_To be written from the real outputs above._

In [ ]:
con.close()